### Market Data (Klines) Transformation
#### Topics Include:
1. Bronze, Silver to Gold
2. Databricks MLOps
3. Databricks Model Serving

In [ ]:
%sql
DROP TABLE IF EXISTS your_database.bronze_klines;
DROP TABLE IF EXISTS your_database.silver_klines;

### Load klines Zip file from S3 (Bronze)

In [ ]:
import boto3
import pandas as pd
import zipfile
import io
from pyspark.sql.functions import col, timestamp_millis

# ==========================================
# 0. Environment Setup and Key Retrieval
# ==========================================
spark.sql("CREATE DATABASE IF NOT EXISTS your_database")

# Use your configured Secret Env
safe_access_key = dbutils.secrets.get(scope="aws_keys", key="access_key")
safe_secret_key = dbutils.secrets.get(scope="aws_keys", key="secret_key")

s3 = boto3.client(
    's3',
    aws_access_key_id=safe_access_key,
    aws_secret_access_key=safe_secret_key,
    region_name="your-region"  # Replace with your actual region
)

bucket = "your-bucket-name"  # Replace with your actual bucket name
prefix_history = "your-prefix/history/"  # Replace with your actual prefix for historical data

# ==========================================
# Stage 1: Decompress S3 ZIP files in memory using Boto3
# ==========================================
print("[INFO] Searching for historical ZIP files in S3...")
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix_history)

all_historical_dfs = []

if 'Contents' in response:
    for obj in response['Contents']:
        if obj['Key'].endswith('.zip'):
            print(f"[PROCESS] Download and decompress: {obj['Key'].split('/')[-1]}")
            zip_obj = s3.get_object(Bucket=bucket, Key=obj['Key'])
            buffer = io.BytesIO(zip_obj['Body'].read())
            
            with zipfile.ZipFile(buffer) as z:
                for csv_filename in z.namelist():
                    with z.open(csv_filename) as f:
                        # Add dtype setting to ensure data reading doesn't fail due to scientific notation
                        df_temp = pd.read_csv(f, header=None, low_memory=False)
                        all_historical_dfs.append(df_temp)

print(f"[INFO] Successfully decompressed {len(all_historical_dfs)} files.")

# ==========================================
# Stage 2: Merge data and convert to PySpark DataFrame
# ==========================================
if len(all_historical_dfs) > 0:
    pdf_history = pd.concat(all_historical_dfs, ignore_index=True)
    
    columns = ["open_time", "open", "high", "low", "close", "volume",
               "close_time", "quote_asset_volume", "number_of_trades",
               "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"]
    pdf_history.columns = columns
    
    # Ensure numeric columns are correctly converted to avoid Spark inferring them as String
    numeric_cols = ["open", "high", "low", "close", "volume", "quote_asset_volume", 
                    "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume"]
    pdf_history[numeric_cols] = pdf_history[numeric_cols].apply(pd.to_numeric, errors='coerce')
    
    # Convert to Spark DataFrame
    df_spark_history = spark.createDataFrame(pdf_history)

    # Stage 3: Append to Bronze Table
    df_spark_history.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("your_database.bronze_klines")  # Replace with your actual database and table name
    
    print(f"[SUCCESS] Bronze Layer write completed. Total records: {pdf_history.shape[0]}")
else:
    print("[WARN] No ZIP files found.")


In [ ]:
display(spark.table("your_database.bronze_klines").orderBy("open_time", ascending=True).limit(10))

### Silver

In [ ]:
from pyspark.sql.functions import col, to_date

# ==========================================
# Stage 1: Read data from Bronze Table
# ==========================================
# Read the Bronze Layer data just written
df_bronze = spark.read.table("your_database.bronze_klines")

# ==========================================
# Stage 2: Data cleaning and transformation (Silver Layer logic)
# ==========================================
# 1. Convert millisecond timestamp to Spark native Timestamp type
# 2. Ensure all price and volume numeric columns have correct type (Double)
# 3. Convert number of trades to integer (Long)
# 4. Drop unnecessary columns (ignore)
# 5. Remove duplicate data (using open_time as unique key)
df_silver = df_bronze \
    .withColumn("open_time_ts", (col("open_time") / 1000000).cast("timestamp")) \
    .withColumn("close_time_ts", (col("close_time") / 1000000).cast("timestamp")) \
    .withColumn("open", col("open").cast("double")) \
    .withColumn("high", col("high").cast("double")) \
    .withColumn("low", col("low").cast("double")) \
    .withColumn("close", col("close").cast("double")) \
    .withColumn("volume", col("volume").cast("double")) \
    .withColumn("quote_asset_volume", col("quote_asset_volume").cast("double")) \
    .withColumn("taker_buy_base_asset_volume", col("taker_buy_base_asset_volume").cast("double")) \
    .withColumn("taker_buy_quote_asset_volume", col("taker_buy_quote_asset_volume").cast("double")) \
    .withColumn("number_of_trades", col("number_of_trades").cast("long")) \
    .drop("ignore") \
    .dropDuplicates(["open_time"]) # Ensure no duplicate dirty data is written for the same K-line start time

# Add a 'date' column as Partition Key to optimize future query performance
df_silver = df_silver.withColumn("trade_date", to_date("open_time_ts"))

# ==========================================
# Stage 3: Write to Silver Table (using Delta Format)
# ==========================================
# Write the cleaned data to Silver Table and partition by trade date (trade_date)
df_silver.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("trade_date") \
    .option("mergeSchema", "true") \
    .saveAsTable("your_database.silver_klines")  # Replace with your actual database and table name

print("[SUCCESS] Silver Layer transformation and write completed.")

In [ ]:
display(spark.table("your_database.silver_klines").orderBy("open_time", ascending=False).limit(6))

### Gold

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, min as _min, last, avg, stddev, when, least, lit,
    datediff, first, lag
)

# ==========================================
# 1. Read Silver Klines data (including warm-up period)
# ==========================================
df_silver = spark.read.table("your_database.silver_klines")  # Replace with your actual database and table name

# Ensure symbol column exists (if not, add default value)
if "symbol" not in df_silver.columns:
    df_silver = df_silver.withColumn("symbol", lit("BTCUSDC"))

# Define signal start time
signal_start_time = "chosen-signal-start-time"  # Replace with your actual signal start time
signal_start_date = signal_start_time[:10] 

# ==========================================
# 2. Define Window operations with Partition
# ==========================================
# Past 14 minutes (calculate short-term ATR)
window_past_14m = Window.partitionBy("symbol").orderBy("open_time_ts").rowsBetween(-13, 0)

# Past 4 hours (calculate market state features, 240 minutes)
window_past_4h = Window.partitionBy("symbol").orderBy("open_time_ts").rowsBetween(-239, 0)

# Exactly 4 hours ago (calculate Momentum)
window_exact_4h_ago = Window.partitionBy("symbol").orderBy("open_time_ts")

# Past 24 hours (calculate large timeframe volatility, 1440 minutes)
window_past_24h = Window.partitionBy("symbol").orderBy("open_time_ts").rowsBetween(-1439, 0)

# Future 72 hours (calculate execution results: 72 * 60 = 4320 minutes)
window_future_72h = Window.partitionBy("symbol").orderBy("open_time_ts").rowsBetween(1, 4320)

# Unbounded Window (used to get the first price on signal day)
window_all_time = Window.partitionBy("symbol").orderBy("open_time_ts")

# ==========================================
# 3. Machine Learning Feature Engineering
# ==========================================
df_gold = df_silver \
    .withColumn("tr_proxy", col("high") - col("low")) \
    .withColumn("atr_14m", avg("tr_proxy").over(window_past_14m)) \
    .withColumn("atr_pct", col("atr_14m") / col("close")) \
    .withColumn("volatility_4h", stddev("close").over(window_past_4h)) \
    .withColumn("ma_4h", avg("close").over(window_past_4h)) \
    .withColumn("bias_to_ma_4h", (col("close") - col("ma_4h")) / col("ma_4h")) \
    .withColumn("atr_24h", avg("tr_proxy").over(window_past_24h)) \
    .withColumn("atr_24h_pct", col("atr_24h") / col("close"))

# [Feature Group A]: Volume Surge (volume burst ratio)
# Ensure the data includes volume column
df_gold = df_gold \
    .withColumn("vol_ma_4h", avg("volume").over(window_past_4h)) \
    .withColumn("volume_surge_ratio", col("volume") / (col("vol_ma_4h") + lit(1e-8)))

# [Feature Group B]: Rejection Tail (lower wick rejection strength)
df_gold = df_gold \
    .withColumn("lower_wick", least(col("open"), col("close")) - col("low")) \
    .withColumn("lower_wick_ratio", col("lower_wick") / (col("tr_proxy") + lit(1e-8)))

# [Feature Group C]: Momentum (ROC_4h - 4-hour price change rate)
df_gold = df_gold \
    .withColumn("close_4h_ago", lag("close", 240).over(window_exact_4h_ago)) \
    .withColumn("roc_4h", (col("close") - col("close_4h_ago")) / col("close_4h_ago"))

# [Feature Group D]: Trend Extension & Time Decay (Days Since Signal)
df_gold = df_gold \
    .withColumn("is_post_signal", when(col("open_time_ts") >= lit(signal_start_time), True).otherwise(False)) \
    .withColumn("signal_price", 
                first(when(col("is_post_signal"), col("close")), ignorenulls=True).over(window_all_time)) \
    .withColumn("trend_extension_pct", 
                when(col("is_post_signal"), (col("close") - col("signal_price")) / col("signal_price")).otherwise(lit(0))) \
    .withColumn("days_since_signal", datediff(col("open_time_ts"), lit(signal_start_date)))

# ==========================================
# 4. Get the lowest price and closing price for the next 72 hours
# ==========================================
df_gold = df_gold \
    .withColumn("future_72h_low", _min("low").over(window_future_72h)) \
    .withColumn("future_72h_close", last("close").over(window_future_72h))

# ==========================================
# 5. Calculate mixed cost for three dynamic ATR pyramid strategies
# ==========================================
df_gold = df_gold.withColumn("cost_A", 
    (0.5 * col("close")) + 
    (0.5 * when(col("future_72h_low") <= col("close") * (1 - col("atr_pct") * 0.5), 
                col("close") * (1 - col("atr_pct") * 0.5)).otherwise(col("future_72h_close")))
).withColumn("cost_B",
    (0.3 * col("close")) +
    (0.4 * when(col("future_72h_low") <= col("close") * (1 - col("atr_pct") * 1.5), 
                col("close") * (1 - col("atr_pct") * 1.5)).otherwise(col("future_72h_close"))) +
    (0.3 * when(col("future_72h_low") <= col("close") * (1 - col("atr_pct") * 2.0), 
                col("close") * (1 - col("atr_pct") * 2.0)).otherwise(col("future_72h_close")))
).withColumn("cost_C",
    (0.1 * col("close")) +
    (0.4 * when(col("future_72h_low") <= col("close") * (1 - col("atr_pct") * 2.0), 
                col("close") * (1 - col("atr_pct") * 2.0)).otherwise(col("future_72h_close"))) +
    (0.5 * when(col("future_72h_low") <= col("close") * (1 - col("atr_pct") * 4.0), 
                col("close") * (1 - col("atr_pct") * 4.0)).otherwise(col("future_72h_close")))
)

# ==========================================
# 6. Generate labels (AutoML goal: find the strategy with the lowest cost within 72 hours)
# ==========================================
df_gold = df_gold.withColumn("min_cost", least("cost_A", "cost_B", "cost_C")) \
    .withColumn("best_strategy_72h", 
                when(col("min_cost") == col("cost_A"), "Aggressive")
                .when(col("min_cost") == col("cost_B"), "Balanced")
                .otherwise("Passive"))

# ==========================================
# 7. Final filtering and saving (Databricks Delta Table)
# ==========================================
df_gold_final = df_gold \
    .filter(col("future_72h_close").isNotNull()) \
    .filter(col("open_time_ts") >= lit(signal_start_time)) \
    .select(
        "open_time_ts", 
        "symbol", 
        "close", 
        # Original features
        "atr_pct", "volatility_4h", "bias_to_ma_4h", 
        # New features
        "atr_24h_pct", "volume_surge_ratio", "lower_wick_ratio", 
        "roc_4h", "trend_extension_pct", "days_since_signal",
        # Target label
        "best_strategy_72h"
    )

# Write to Delta Lake (enable mergeSchema to support newly added feature columns)
df_gold_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("your_database.gold_late_capital_72h")  # Replace with your actual database and table name
print("[SUCCESS] Gold Layer feature engineering and write completed.")

In [ ]:
display(spark.table("your_database.gold_late_capital_72h").orderBy("open_time_ts", ascending=True).limit(20))

### AutoML

In [ ]:
%pip install "flaml[automl]" --quiet

In [ ]:
dbutils.library.restartPython()

In [ ]:
import mlflow
from flaml import AutoML
import pandas as pd

# 1. Load your Gold Table and convert it to Pandas (Serverless can handle this)
train_df = spark.read.table("your_database.gold_late_capital_72h")  # Replace with your actual database and table name
train_pdf = train_df.toPandas()

# 2. Make sure the data is sorted by time (this is very important for time-series prediction)
train_pdf = train_pdf.sort_values('open_time_ts').reset_index(drop=True)

# 3. Split features (X) and label (y)
# Note: remove the label, time column, and any ID columns that should not be used as features (e.g., symbol)
X_train = train_pdf.drop(columns=['best_strategy_72h', 'open_time_ts', 'symbol']) 
y_train = train_pdf['best_strategy_72h']

print("[INFO] Starting FLAML AutoML (Serverless workaround)...")

# 4. Configure and initialize FLAML
automl = AutoML()
automl_settings = {
    "time_budget": 600,         # Limit training time to 10 minutes (600 seconds)
    "metric": 'macro_f1',       # Evaluation metric
    "task": 'classification',   # Task type
    "eval_method": "holdout",   # For time-series data, typically use the last segment as validation
    "split_type": "time",       # 👑 Key: tell FLAML this is time-series data, do NOT shuffle!
    "log_file_name": "late_capital_automl.log"
}

# Start training
automl.fit(X_train=X_train, y_train=y_train, **automl_settings)

print("\n[SUCCESS] FLAML AutoML training completed!")
print(f"👑 Best model algorithm: {automl.best_estimator}")
print(f"🎯 Best hyperparameters: {automl.best_config}")
print(f"📈 Best validation F1 Score: {1 - automl.best_loss:.4f}")  # Lower loss is better in FLAML, so 1 - loss approximates the score

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

# 1. Define the model name to register
model_name = "crypto_execution_optimizer"

# 2. Start MLflow experiment tracking
# You can change this to your own workspace path, or just let it use the default path
mlflow.set_experiment("Path/To/Your/Experiment")  # Replace with your actual experiment name or path

with mlflow.start_run(run_name="FLAML_ExtraTree_Run") as run:
    
    # Log your hyperparameters and metrics
    mlflow.log_params(automl.best_config)
    mlflow.log_metric("val_macro_f1", 1 - automl.best_loss)
    
    # 👑 Key: infer the model signature
    # This step is VERY important for Model Serving later,
    # as the API needs to know the expected input data format
    sample_input = X_train.head(5)
    sample_output = automl.predict(sample_input)
    signature = infer_signature(sample_input, sample_output)
    
    # Extract the underlying scikit-learn model from FLAML
    best_sklearn_model = automl.model.estimator
    
    # Log the model to MLflow and "automatically register" it to the Model Registry
    model_info = mlflow.sklearn.log_model(
        sk_model=best_sklearn_model,
        artifact_path="model",
        signature=signature,
        registered_model_name=model_name
    )